## Advanced chunking strategies provided by Amazon Bedrock Knowledge Bases

In this notebook, we will use a **single knowledge base** and cycle through the following chunking options supported by Amazon Bedrock Knowledge Bases: 
1. Fixed chunking
2. Semantic chunking
3. Hierarchical chunking
4. Custom chunking using Lambda function

Chunking breaks down the text into smaller segments before embedding. The chunking strategy is set at the **data source** level. Rather than creating a separate knowledge base per strategy, we create one KB, add a data source for each strategy, evaluate it using RAGAS, **delete that data source**, then add the next. This keeps the vector index clean so retrieval results are never mixed across strategies.

We will use a synthetic 10K report as data for a fictitious company called `Octank Financial` to demo the solution. The focus will be on improving the quality of search results which in turn will improve the accuracy of responses generated by the foundation model.

## 1. Import the needed libraries
First step is to install the pre-requisites packages.

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --no-deps --quiet
%pip install -r ../requirements.txt --upgrade --quiet

In [ ]:
# restart kernel
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import botocore
botocore.__version__

In [ ]:
import os
import sys
import time
import boto3
import logging
import pprint
import json

# Set the path to import module
from pathlib import Path
current_path = Path().resolve()
current_path = current_path.parent
if str(current_path) not in sys.path:
    sys.path.append(str(current_path))
# Print sys.path to verify
# print(sys.path)

from utils.knowledge_base import BedrockKnowledgeBase

In [ ]:
#Clients
s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region =  session.region_name
account_id = sts_client.get_caller_identity()["Account"]
bedrock_agent_client = boto3.client('bedrock-agent')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime') 
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)
region, account_id

In [ ]:
import time

# Get the current timestamp
current_time = time.time()

# Format the timestamp as a string
timestamp_str = time.strftime("%Y%m%d%H%M%S", time.localtime(current_time))[-7:]
# Create the suffix using the timestamp
suffix = f"{timestamp_str}"
knowledge_base_name_standard = 'standard-kb'
knowledge_base_description = "Knowledge Base containing complex PDF."
bucket_name = f'{knowledge_base_name_standard}-{suffix}'
intermediate_bucket_name = f'{knowledge_base_name_standard}-intermediate-{suffix}'
lambda_function_name = f'{knowledge_base_name_standard}-custom-lambda-{suffix}'
foundation_model = "us.amazon.nova-2-lite-v1:0"

data_source=[{"type": "S3", "bucket_name": bucket_name}]

## 2 - Create knowledge bases with fixed chunking strategy
Let's start by creating a [Amazon Bedrock Knowledge Bases](https://aws.amazon.com/bedrock/knowledge-bases/) to store the restaurant menus. Knowledge Bases allow you to integrate with different vector databases including [Amazon OpenSearch Serverless](https://aws.amazon.com/opensearch-service/features/serverless/), [Amazon Aurora](https://aws.amazon.com/rds/aurora/), [Pinecone](http://app.pinecone.io/bedrock-integration), [Redis Enterprise]() and [MongoDB Atlas](). For this example, we will integrate the knowledge base with Amazon OpenSearch Serverless. To do so, we will use the helper class `BedrockKnowledgeBase` which will create the knowledge base and all of its pre-requisites:
1. IAM roles and policies
2. S3 bucket
3. Amazon OpenSearch Serverless encryption, network and data access policies
4. Amazon OpenSearch Serverless collection
5. Amazon OpenSearch Serverless vector index
6. Knowledge base
7. Knowledge base data source

First we will create a knowledge base using fixed chunking strategy followed by hierarchical chunking strategy. 

Parameter values: 
```
"chunkingStrategy": "FIXED_SIZE | NONE | HIERARCHICAL | SEMANTIC"
```

In [ ]:
knowledge_base_standard = BedrockKnowledgeBase(
    kb_name=f'{knowledge_base_name_standard}-{suffix}',
    kb_description=knowledge_base_description,
    data_sources=data_source,
    chunking_strategy = "FIXED_SIZE", 
    suffix = f'{suffix}-f'
)

## 2.1 Upload the dataset to Amazon S3
Now that we have created the knowledge base, let's populate it with the `Octank financial 10K` report dataset. The Knowledge Base data source expects the data to be available on the S3 bucket connected to it and changes on the data can be syncronized to the knowledge base using the `StartIngestionJob` API call. In this example we will use the [boto3 abstraction](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent/client/start_ingestion_job.html) of the API, via our helper classe. 

Let's first upload the menu's data available on the `dataset` folder to s3.

In [ ]:
import os

def upload_directory(path, bucket_name):
    for root, dirs, files in os.walk(path):
        for file in files:
            file_to_upload = os.path.join(root, file)
            if file not in ["LICENSE", "NOTICE", "README.md"]:
                print(f"uploading file {file_to_upload} to {bucket_name}")
                s3_client.upload_file(file_to_upload, bucket_name, file)
            else:
                print(f"Skipping file {file_to_upload}")

upload_directory("../synthetic_dataset", bucket_name)


Now we start the ingestion job.

In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base_standard.start_ingestion_job()

Finally we save the Knowledge Base Id to test the solution at a later stage. 

In [ ]:
kb_id_standard = knowledge_base_standard.get_knowledge_base_id()

# Get the data source ID for the fixed chunking strategy
ds_response = bedrock_agent_client.list_data_sources(
    knowledgeBaseId=kb_id_standard,
    maxResults=100
)
ds_id_fixed = ds_response['dataSourceSummaries'][0]['dataSourceId']
print(f"Knowledge Base ID: {kb_id_standard}")
print(f"Fixed chunking data source ID: {ds_id_fixed}")

In [ ]:
# Persist the shared KB ID and S3 bucket name for downstream notebooks
kb_id = kb_id_standard
%store kb_id
%store bucket_name
print(f"Shared KB ID stored: {kb_id}")
print(f"Shared S3 bucket stored: {bucket_name}")

### 2.2 Test the Knowledge Base with Fixed Chunking
Now the Knowlegde Base is available we can test it out using the [**retrieve**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve.html) and [**retrieve_and_generate**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html) functions. 

#### Testing Knowledge Base with Retrieve and Generate API

Let's first test the knowledge base using the retrieve and generate API. With this API, Bedrock takes care of retrieving the necessary references from the knowledge base and generating the final answer using a foundation model from Bedrock.

query = `Provide a summary of consolidated statements of cash flows of Octank Financial for the fiscal years ended December 31, 2019.`

The right response for this query as per ground truth QA pair is: 

```
The cash flow statement for Octank Financial in the year ended December 31, 2019 reveals the following:
- Cash generated from operating activities amounted to $710 million, which can be attributed to a $700 million profit and non-cash charges such as depreciation and amortization.
- Cash outflow from investing activities totaled $240 million, with major expenditures being the acquisition of property, plant, and equipment ($200 million) and marketable securities ($60 million), partially offset by the sale of property, plant, and equipment ($40 million) and maturing marketable securities ($20 million).
- Financing activities resulted in a cash inflow of $350 million, stemming from the issuance of common stock ($200 million) and long-term debt ($300 million), while common stock repurchases ($50 million) and long-term debt payments ($100 million) reduced the cash inflow. 
Overall, Octank Financial experienced a net cash enhancement of $120 million in 2019, bringing their total cash and cash equivalents to $210 million.
```

In [ ]:
query = "Provide a summary of consolidated statements of cash flows of Octank Financial for the fiscal years ended December 31, 2019."

In [ ]:
time.sleep(20)
response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id_standard,
            "modelArn": foundation_model,
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)

print(response['output']['text'],end='\n'*2)

As you can see, with the retrieve and generate API we get the final response directly, now let's observe the citations for `RetreiveAndGenerate` API. Since, our primary focus on this notebook is to observe the retrieved chunks and citations returned by the model while generating the response. When we provide the relevant context to the foundation model alongwith the query, it will most likely generate the high quality response. 

In [ ]:
def citations_rag_print(response_ret):
#structure 'retrievalResults': list of contents. Each list has content, location, score, metadata
    for num,chunk in enumerate(response_ret,1):
        print(f'Chunk {num}: ',chunk['content']['text'],end='\n'*2)
        print(f'Chunk {num} Location: ',chunk['location'],end='\n'*2)
        print(f'Chunk {num} Metadata: ',chunk['metadata'],end='\n'*2)

In [ ]:
response_standard = response['citations'][0]['retrievedReferences']
print("# of citations or chunks used to generate the response: ", len(response_standard))
citations_rag_print(response_standard)

Let's now inspect the source information from the knowledge base with the retrieve API.

#### Testing Knowledge Base with Retrieve API
If you need an extra layer of control, you can retrieve the chunks that best match your query using the retrieve API. In this setup, we can configure the desired number of results and control the final answer with your own application logic. The API then provides you with the matching content, its S3 location, the similarity score and the chunk metadata.

In [ ]:
def response_print(response_ret):
#structure 'retrievalResults': list of contents. Each list has content, location, score, metadata
    for num,chunk in enumerate(response_ret['retrievalResults'],1):
        print(f'Chunk {num}: ',chunk['content']['text'],end='\n'*2)
        print(f'Chunk {num} Location: ',chunk['location'],end='\n'*2)
        print(f'Chunk {num} Score: ',chunk['score'],end='\n'*2)
        print(f'Chunk {num} Metadata: ',chunk['metadata'],end='\n'*2)


In [ ]:
response_standard_ret = bedrock_agent_runtime_client.retrieve(
    knowledgeBaseId=kb_id_standard, 
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults":5,
        } 
    },
    retrievalQuery={
        'text': query
    }
)

print("# of retrieved results: ", len(response_standard_ret['retrievalResults']))
response_print(response_standard_ret)

In [ ]:
# Delete fixed data source and wait for KB to stabilise before adding next
print(f"\nDeleting fixed chunking data source: {ds_id_fixed}")
bedrock_agent_client.delete_data_source(dataSourceId=ds_id_fixed, knowledgeBaseId=kb_id_standard)
print("Deleted. Waiting 30s for KB to stabilise before adding hierarchical data source...")
time.sleep(30)

## 3. Add data source with hierarchical chunking strategy

**Concept**

Hierarchical chunking organizes your data into a hierarchical structure, allowing for more granular and efficient retrieval based on the inherent relationships within your data. It creates parent chunks (larger context windows) and child chunks (more specific passages), maintaining the parent-child relationship during retrieval.

After performing semantic search on child chunks, the API returns the corresponding parent chunk — providing larger, more comprehensive context to the foundation model.

><br>          
>Note:
>In hierarchical chunking, parent chunks are returned and search is performed on children chunks, therefore, you might see fewer search results returned as one parent can have multiple children.
> <br></br>       

**Parameter values:** 
```
"chunkingStrategy": "HIERARCHICAL"
```

In [ ]:
# Add hierarchical chunking data source to the existing knowledge base
hierarchical_chunking_config = {
    "chunkingStrategy": "HIERARCHICAL",
    "hierarchicalChunkingConfiguration": {
        "levelConfigurations": [{"maxTokens": 1500}, {"maxTokens": 300}],
        "overlapTokens": 60
    }
}

ds_hierarchical_response = bedrock_agent_client.create_data_source(
    knowledgeBaseId=kb_id_standard,
    name=f'hierarchical-ds-{suffix}',
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": f"arn:aws:s3:::{bucket_name}"
        }
    },
    vectorIngestionConfiguration={
        "chunkingConfiguration": hierarchical_chunking_config
    }
)
ds_id_hierarchical = ds_hierarchical_response['dataSource']['dataSourceId']
print(f"Hierarchical data source ID: {ds_id_hierarchical}")

Now start the ingestion job. Since, we are using the same documents as used for fixed chunking, we are skipping the step to upload documents to s3 bucket. 

In [ ]:
# Start ingestion for hierarchical data source and wait for completion
start_job_response = bedrock_agent_client.start_ingestion_job(
    knowledgeBaseId=kb_id_standard,
    dataSourceId=ds_id_hierarchical
)

job = start_job_response["ingestionJob"]
print(f"Ingestion job started: {job['ingestionJobId']}")

while job['status'] not in ["COMPLETE", "FAILED", "STOPPED"]:
    time.sleep(10)
    get_job_response = bedrock_agent_client.get_ingestion_job(
        knowledgeBaseId=kb_id_standard,
        dataSourceId=ds_id_hierarchical,
        ingestionJobId=job["ingestionJobId"]
    )
    job = get_job_response["ingestionJob"]
    print(f"Status: {job['status']}")

print(f"\nIngestion complete. Status: {job['status']}")

### 3.1 Test the Knowledge Base with Hierarchical Chunking

In [ ]:
time.sleep(20)
response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id_standard,
            "modelArn": foundation_model,
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)

print(response['output']['text'],end='\n'*2)

#### Testing Knowledge Base with Retrieve API

In [ ]:
response_hierarchical_ret = bedrock_agent_runtime_client.retrieve(
    knowledgeBaseId=kb_id_standard, 
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults":5,
        } 
    },
    retrievalQuery={
        'text': query
    }
)

print("# of retrieved results: ", len(response_hierarchical_ret['retrievalResults']))
response_print(response_hierarchical_ret)

><br>
> Note:
>As you can see in the above response, that the `retrieve` API returned only 3 search results or chunks although 5 were passed in the request. The reason is that with `hiearchical` chunking, parent chunks are returned by the API whereas search is performed on `children chunks` and one `parent chunk` can have multiple `children chunks`. Therefore, response returned only 3 chunks while the search was performed on 5 `children chunks`.
><br></br>

In [ ]:
# Delete hierarchical data source and wait for KB to stabilise
print(f"\nDeleting hierarchical data source: {ds_id_hierarchical}")
bedrock_agent_client.delete_data_source(dataSourceId=ds_id_hierarchical, knowledgeBaseId=kb_id_standard)
print("Deleted. Waiting 30s for KB to stabilise before adding semantic data source...")
time.sleep(30)

## 4. Add data source with semantic chunking strategy

**Concept**

Semantic chunking analyzes the relationships within a text and divides it into meaningful and complete chunks, which are derived based on the semantic similarity calculated by the embedding model. This approach preserves the information's integrity during retrieval, helping to ensure accurate and contextually appropriate results.

**Benefits**

- By focusing on the text's meaning and context, semantic chunking significantly improves the quality of retrieval.
- It is beneficial for chunking documents where contextual boundaries aren't clear — for example, legal documents or technical manuals.

**Parameter values:**
 
```
"chunkingStrategy": "SEMANTIC"
```

In [ ]:
# Add semantic chunking data source to the existing knowledge base
semantic_chunking_config = {
    "chunkingStrategy": "SEMANTIC",
    "semanticChunkingConfiguration": {
        "maxTokens": 300,
        "bufferSize": 1,
        "breakpointPercentileThreshold": 95
    }
}

ds_semantic_response = bedrock_agent_client.create_data_source(
    knowledgeBaseId=kb_id_standard,
    name=f'semantic-ds-{suffix}',
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {"bucketArn": f"arn:aws:s3:::{bucket_name}"}
    },
    vectorIngestionConfiguration={"chunkingConfiguration": semantic_chunking_config}
)
ds_id_semantic = ds_semantic_response['dataSource']['dataSourceId']
print(f"Semantic data source ID: {ds_id_semantic}")

Now start the ingestion job. Since, we are using the same documents as used for fixed chunking, we are skipping the step to upload documents to s3 bucket. 

In [ ]:
# Start ingestion for semantic data source and wait for completion
start_job_response = bedrock_agent_client.start_ingestion_job(
    knowledgeBaseId=kb_id_standard,
    dataSourceId=ds_id_semantic
)

job = start_job_response["ingestionJob"]
print(f"Ingestion job started: {job['ingestionJobId']}")

while job['status'] not in ["COMPLETE", "FAILED", "STOPPED"]:
    time.sleep(10)
    get_job_response = bedrock_agent_client.get_ingestion_job(
        knowledgeBaseId=kb_id_standard,
        dataSourceId=ds_id_semantic,
        ingestionJobId=job["ingestionJobId"]
    )
    job = get_job_response["ingestionJob"]
    print(f"Status: {job['status']}")

print(f"\nIngestion complete. Status: {job['status']}")

### 4.1 Test the Knowledge Base WITH Semantic Chunking

In [ ]:
time.sleep(20)

response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id_standard,
            "modelArn": foundation_model,
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)

print(response['output']['text'],end='\n'*2)

In [ ]:
response_semantic = response['citations'][0]['retrievedReferences']
print("# of citations or chunks used to generate the response: ", len(response_semantic))
citations_rag_print(response_semantic)

In [ ]:
# Delete semantic data source and wait for KB to stabilise
print(f"\nDeleting semantic data source: {ds_id_semantic}")
bedrock_agent_client.delete_data_source(dataSourceId=ds_id_semantic, knowledgeBaseId=kb_id_standard)
print("Deleted. Waiting 30s for KB to stabilise before adding custom data source...")
time.sleep(30)

## 5. Add data source with custom chunking using Lambda Functions

When creating a data source for Amazon Bedrock Knowledge Bases, you can connect a Lambda function to specify your custom chunking logic. During ingestion, Knowledge Bases will invoke the Lambda function and store the input/output in an intermediate S3 bucket.

> <br>
> Note: Lambda function with KB can be used for adding custom chunking logic as well as processing your chunks — for example, adding chunk-level metadata. In this example we are focusing on using a Lambda function for custom chunking logic.
> <br></br>

### 5.1 Create the Lambda Function

We will create a lambda function which will have code for custom chunking. To do so we will:

1. Create the `lambda_function.py` file which contains the logic for custom chunking.
2. Create the IAM role for our Lambda function.
3. Create the lambda function with the required permissions.
4. Add the data source to the **existing** knowledge base pointing to this Lambda.

#### Create the function code
 Let's create the lambda function tha implements the functions for `reading your file from intermediate bucket`, `process the contents with custom chunking logic` and `write the output back to s3 bucket`. 

In [ ]:
%%writefile lambda_function.py
import json
from abc import abstractmethod, ABC
from typing import List
import boto3
import logging
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

class Chunker(ABC):
    @abstractmethod
    def chunk(self, text: str) -> List[str]:
        raise NotImplementedError()

class SimpleChunker(Chunker):
    def chunk(self, text: str) -> List[str]:
        words = text.split()
        return [' '.join(words[i:i+100]) for i in range(0, len(words), 100)]

def lambda_handler(event, context):
    logger.debug('input={}'.format(json.dumps(event)))
    s3 = boto3.client('s3')
    input_files = event.get('inputFiles')
    input_bucket = event.get('bucketName')
    if not all([input_files, input_bucket]):
        raise ValueError("Missing required input parameters")
    output_files = []
    chunker = SimpleChunker()
    for input_file in input_files:
        content_batches = input_file.get('contentBatches', [])
        file_metadata = input_file.get('fileMetadata', {})
        original_file_location = input_file.get('originalFileLocation', {})
        processed_batches = []
        for batch in content_batches:
            input_key = batch.get('key')
            if not input_key:
                raise ValueError("Missing uri in content batch")
            file_content = read_s3_file(s3, input_bucket, input_key)
            chunked_content = process_content(file_content, chunker)
            output_key = f"Output/{input_key}"
            write_to_s3(s3, input_bucket, output_key, chunked_content)
            processed_batches.append({'key': output_key})
        output_files.append({
            'originalFileLocation': original_file_location,
            'fileMetadata': file_metadata,
            'contentBatches': processed_batches
        })
    return {'outputFiles': output_files}

def read_s3_file(s3_client, bucket, key):
    response = s3_client.get_object(Bucket=bucket, Key=key)
    return json.loads(response['Body'].read().decode('utf-8'))

def write_to_s3(s3_client, bucket, key, content):
    s3_client.put_object(Bucket=bucket, Key=key, Body=json.dumps(content))

def process_content(file_content: dict, chunker: Chunker) -> dict:
    chunked_content = {'fileContents': []}
    for content in file_content.get('fileContents', []):
        chunks = chunker.chunk(content.get('contentBody', ''))
        for chunk in chunks:
            chunked_content['fileContents'].append({
                'contentType': content.get('contentType', ''),
                'contentMetadata': content.get('contentMetadata', {}),
                'contentBody': chunk
            })
    return chunked_content

For custom chunking, we add a data source with a `customTransformationConfiguration` that points to our Lambda function. The data source configuration looks like:

```
{
   "vectorIngestionConfiguration": {
    "customTransformationConfiguration": { 
         "intermediateStorage": { 
            "s3Location": { 
               "uri": "s3://intermediate-bucket/"
            }
         },
         "transformations": [
            {
               "transformationFunction": {
                  "transformationLambdaConfiguration": {
                     "lambdaArn": "arn:aws:lambda:..."
                  }
               },
               "stepToApply": "POST_CHUNKING"
            }
         ]
      },
      "chunkingConfiguration": {
         "chunkingStrategy": "NONE"
      }
   }
}
```

We will manually create the intermediate S3 bucket, IAM role, and Lambda function, then add the data source to our existing knowledge base.

In [ ]:
import zipfile
from io import BytesIO

lambda_client = boto3.client('lambda')
iam_client = boto3.client('iam')

# Create intermediate S3 bucket for Lambda I/O
try:
    if region == "us-east-1":
        s3_client.create_bucket(Bucket=intermediate_bucket_name)
    else:
        s3_client.create_bucket(
            Bucket=intermediate_bucket_name,
            CreateBucketConfiguration={'LocationConstraint': region}
        )
    print(f"Created intermediate bucket: {intermediate_bucket_name}")
except s3_client.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket already exists: {intermediate_bucket_name}")

# Create Lambda execution role
lambda_role_name = f'BedrockKBCustomChunkingRole-{suffix}'
assume_role_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]
})

try:
    role_response = iam_client.create_role(
        RoleName=lambda_role_name,
        AssumeRolePolicyDocument=assume_role_policy,
        Description='Role for custom chunking Lambda'
    )
    lambda_role_arn = role_response['Role']['Arn']
except iam_client.exceptions.EntityAlreadyExistsException:
    lambda_role_arn = iam_client.get_role(RoleName=lambda_role_name)['Role']['Arn']

for policy_arn in [
    'arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole',
    'arn:aws:iam::aws:policy/AmazonS3FullAccess'
]:
    iam_client.attach_role_policy(RoleName=lambda_role_name, PolicyArn=policy_arn)

time.sleep(10)

# Package and deploy Lambda
zip_buffer = BytesIO()
with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write('lambda_function.py')
zip_buffer.seek(0)

try:
    lambda_response = lambda_client.create_function(
        FunctionName=lambda_function_name,
        Runtime='python3.12',
        Role=lambda_role_arn,
        Handler='lambda_function.lambda_handler',
        Code={'ZipFile': zip_buffer.read()},
        Timeout=900,
        MemorySize=256
    )
    lambda_arn = lambda_response['FunctionArn']
except lambda_client.exceptions.ResourceConflictException:
    lambda_arn = lambda_client.get_function(FunctionName=lambda_function_name)['Configuration']['FunctionArn']

print(f"Lambda ARN: {lambda_arn}")

# Allow Bedrock to invoke the Lambda
try:
    lambda_client.add_permission(
        FunctionName=lambda_function_name,
        StatementId='AllowBedrockInvoke',
        Action='lambda:InvokeFunction',
        Principal='bedrock.amazonaws.com',
        SourceArn=f'arn:aws:bedrock:{region}:{account_id}:knowledge-base/{kb_id_standard}'
    )
except lambda_client.exceptions.ResourceConflictException:
    print("Permission already exists")

# Add custom chunking data source to the existing knowledge base
ds_custom_response = bedrock_agent_client.create_data_source(
    knowledgeBaseId=kb_id_standard,
    name=f'custom-ds-{suffix}',
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {"bucketArn": f"arn:aws:s3:::{bucket_name}"}
    },
    vectorIngestionConfiguration={
        "customTransformationConfiguration": {
            "intermediateStorage": {"s3Location": {"uri": f"s3://{intermediate_bucket_name}/"}},
            "transformations": [{
                "transformationFunction": {
                    "transformationLambdaConfiguration": {"lambdaArn": lambda_arn}
                },
                "stepToApply": "POST_CHUNKING"
            }]
        },
        "chunkingConfiguration": {"chunkingStrategy": "NONE"}
    }
)
ds_id_custom = ds_custom_response['dataSource']['dataSourceId']
print(f"Custom chunking data source ID: {ds_id_custom}")

Now start the ingestion job. 

In [ ]:
knowledge_base_standard.ensure_custom_chunking_permissions(
    kb_id=kb_id_standard,
    bucket_name=bucket_name,
    intermediate_bucket_name=intermediate_bucket_name,
    lambda_function_name=lambda_function_name
)

# Start ingestion for custom data source and wait for completion
start_job_response = bedrock_agent_client.start_ingestion_job(
    knowledgeBaseId=kb_id_standard,
    dataSourceId=ds_id_custom
)

job = start_job_response["ingestionJob"]
print(f"Ingestion job started: {job['ingestionJobId']}")

while job['status'] not in ["COMPLETE", "FAILED", "STOPPED"]:
    time.sleep(10)
    get_job_response = bedrock_agent_client.get_ingestion_job(
        knowledgeBaseId=kb_id_standard,
        dataSourceId=ds_id_custom,
        ingestionJobId=job["ingestionJobId"]
    )
    job = get_job_response["ingestionJob"]
    print(f"Status: {job['status']}")

print(f"Ingestion complete. Status: {job['status']}")


### 5.2 Test the Knowledge Base with Custom Chunking strategy

In [ ]:
time.sleep(10)

response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id_standard,
            "modelArn": foundation_model,
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)

print(response['output']['text'],end='\n'*2)

In [ ]:
response_custom = response['citations'][0]['retrievedReferences']
print("# of citations: ", len(response_custom))
citations_rag_print(response_custom)

In [ ]:
# Delete custom data source — all strategies evaluated
print(f"\nDeleting custom data source: {ds_id_custom}")
bedrock_agent_client.delete_data_source(dataSourceId=ds_id_custom, knowledgeBaseId=kb_id_standard)
print("Deleted. All strategies evaluated — proceed to comparison.")

## 7. Cleanup

> **⚠️ WARNING — Run cleanup only after completing the entire module**
>
> Run the cleanup cell below **only after** you have finished all five downstream notebooks in this module:
>
> 1. `autogenerated_metadata_filters.ipynb`
> 2. `csv_metadata_customization.ipynb`
> 3. `metadata_filtering.ipynb`
> 4. `query_reformulation.ipynb`
> 5. `re_ranking_using_kb.ipynb`
>
> The shared Knowledge Base provisioned in this notebook is reused by every downstream notebook. Deleting it before the downstream notebooks finish will break their `retrieve` and `retrieve_and_generate` calls. This cleanup also tears down the shared S3 bucket, IAM roles, OpenSearch Serverless collection, and the comparison-only Knowledge Bases (hierarchical, semantic, custom) created earlier in this notebook.

In [ ]:
# WARNING: Run this cell ONLY after completing all notebooks in the
# 02-optimizing-accuracy-retrieved-results module.

# print("=== Deleting shared Knowledge Base ===")
# knowledge_base_standard.delete_kb(delete_s3_bucket=True, delete_iam_roles_and_policies=True)

# print("\n=== Cleaning up Lambda and intermediate bucket ===")
# try:
#     lambda_client.delete_function(FunctionName=lambda_function_name)
#     print(f"  Deleted Lambda: {lambda_function_name}")
# except Exception as e:
#     print(f"  Lambda: {e}")

# try:
#     for policy_arn in [
#         'arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole',
#         'arn:aws:iam::aws:policy/AmazonS3FullAccess'
#     ]:
#         iam_client.detach_role_policy(RoleName=lambda_role_name, PolicyArn=policy_arn)
#     iam_client.delete_role(RoleName=lambda_role_name)
#     print(f"  Deleted IAM role: {lambda_role_name}")
# except Exception as e:
#     print(f"  IAM role: {e}")

# try:
#     paginator = s3_client.get_paginator('list_objects_v2')
#     for page in paginator.paginate(Bucket=intermediate_bucket_name):
#         for obj in page.get('Contents', []):
#             s3_client.delete_object(Bucket=intermediate_bucket_name, Key=obj['Key'])
#     s3_client.delete_bucket(Bucket=intermediate_bucket_name)
#     print(f"  Deleted intermediate bucket: {intermediate_bucket_name}")
# except Exception as e:
#     print(f"  Intermediate bucket: {e}")

# print("\n=== Cleanup complete ===")